# Awal

In [50]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# import numpy as np

# import os
# from preprocessing.image_utils import extract_features

# def extract_and_save_npy(image_dir, model_cnn, output_dir):
        
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
        
#     images_file = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

#     features = []
#     names = []

#     for img_name in (images_file):
#         img_path = os.path.join(image_dir, img_name)
#         feat = extract_features(model_cnn, img_path)

#         features.append(feat)
#         names.append(img_name)

#     features_matrix = np.array(features)
#     names_array = np.array(names)

#     np.save(os.path.join(output_dir, "features.npy"), features_matrix)
#     np.save(os.path.join(output_dir, "image_names.npy"), names_array)

#     print("Selesai! Fitur dan Nama Gambar telah disimpan.")


In [14]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
import numpy as np

image_dir = "../../dataset/Images"
output_dir = "../../images_feature"

base_model = InceptionV3(weights='imagenet')
model_cnn = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

features_path = os.path.join(output_dir, "features.npy")
names_path = os.path.join(output_dir, "image_names.npy")

# if not os.path.exists(features_path) or not os.path.exists(names_path):
#     print("Memulai proses ekstraksi...")
#     extract_and_save_npy(image_dir, model_cnn, output_dir)
# else:
#     print("Load Feature...")


features_matrix = np.load(features_path)
image_names = np.load(names_path)

image_features_map = dict(zip(image_names, features_matrix))


#### Captions

In [16]:
from utils.text_utils import CaptionPreprocessor

text_util = CaptionPreprocessor()
text_util.build_vocabulary("../../RNN_dataset/captions.txt")
map_image_cap = text_util.get_image_to_captions_mapping("../../RNN_dataset/captions.txt")
data = {}

for image in map_image_cap:
    captions = map_image_cap[image]
    seq = text_util.texts_to_sequences(captions=captions)
    padded_seq = text_util.pad_sequences(sequences=seq)
    data[image] = padded_seq




### Bagian 3

In [17]:
import tensorflow as tf
import time
import pandas as pd
from utils.train_utils import DataGenerator
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, SimpleRNN, Concatenate, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Masking

def build_model(lstm: bool, layers: int, hidden_state_count: int, vocab_size: int, sequence_length: int = 35):
    image_input = Input(shape=(2048,), name="Image_Input")
    caption_input = Input(shape=(sequence_length,), name="Caption_Input")

    image_projection = Dense(256, activation='relu')(image_input)
    image_projection = Reshape((1, 256))(image_projection)
    image_projection = Masking(mask_value=0.0)(image_projection) 

    caption_embedding = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input)

    merged = Concatenate(axis=1)([image_projection, caption_embedding])

    x = merged
    for i in range(layers):
        if lstm:
            x = LSTM(hidden_state_count, return_sequences=True, name=f"LSTM_Layer_{i+1}")(x)
        else:
            x = SimpleRNN(hidden_state_count, return_sequences=True, name=f"RNN_Layer_{i+1}")(x)

    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    
    outputs = Dense(vocab_size, activation='softmax', name="Output_Layer")(x)

    model = Model(inputs=[image_input, caption_input], outputs=outputs)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.summary()
    return model

In [18]:
variations = [
    {"layers": 1, "hidden": 128, "name": "Shallow_Small"},
    {"layers": 2, "hidden": 128, "name": "Deep_Small"},
    {"layers": 3, "hidden": 128, "name": "VeryDeep_Small"},
    {"layers": 1, "hidden": 256, "name": "Shallow_Mid"},
    {"layers": 1, "hidden": 512, "name": "Shallow_Large"},
    {"layers": 2, "hidden": 512, "name": "Deep_Large"},
]

# Tentukan parameter umum
VOCAB_SIZE = text_util.vocab_size
SEQ_LEN = text_util.sequence_length
EPOCHS = 5
BATCH_SIZE = 64

In [19]:
all_images = list(data.keys())
np.random.seed(42)
np.random.shuffle(all_images)

train_keys = all_images[:6000]
val_keys   = all_images[6000:7000]
test_keys  = all_images[7000:8000]

In [20]:
train_data = {k: data[k] for k in train_keys}
val_data   = {k: data[k] for k in val_keys}
test_data  = {k: data[k] for k in test_keys}

In [21]:
history_logs = []
os.makedirs("weights", exist_ok=True)

for is_lstm in [False, True]:
    arch_name = "LSTM" if is_lstm else "SimpleRNN"
    
    for var in variations:
        model_name = f"{arch_name}_{var['name']}_L{var['layers']}_H{var['hidden']}"
        print(f"\n{'='*50}\nMemulai Training: {model_name}\n{'='*50}")
        
        model = build_model(
            lstm=is_lstm, 
            layers=var['layers'], 
            hidden_state_count=var['hidden'], 
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN
        )
        
        train_gen = DataGenerator(
            mapping_data=train_data,         # <-- ganti dari data ke train_data
            image_features=image_features_map, 
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN,
            batch_size=BATCH_SIZE
        ).generate()

        val_gen = DataGenerator(             # <-- tambahkan val_gen
            mapping_data=val_data,
            image_features=image_features_map,
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN,
            batch_size=BATCH_SIZE
        ).generate()
        
        start_time = time.time()
        history = model.fit(
            train_gen, 
            steps_per_epoch=len(train_data) // BATCH_SIZE,   # <-- sesuaikan
            validation_data=val_gen,                          # <-- tambahkan
            validation_steps=len(val_data) // BATCH_SIZE,    # <-- tambahkan
            epochs=EPOCHS,
            verbose=1
        )
        end_time = time.time()
        
        model.save_weights(f"weights/{model_name}.h5")
        
        duration = end_time - start_time
        history_logs.append({
            "model_type": arch_name,
            "variation_name": var['name'],
            "layers": var['layers'],
            "hidden_state": var['hidden'],
            "final_loss": history.history['loss'][-1],
            "final_val_loss": history.history['val_loss'][-1],   # <-- tambahkan
            "history_loss": history.history['loss'],             # <-- tambahkan untuk grafik
            "history_val_loss": history.history['val_loss'],     # <-- tambahkan untuk grafik
            "training_time_sec": duration
        })

df_results = pd.DataFrame(history_logs)
df_results.to_csv("hasil_variasi_model.csv", index=False)
print("\nEksperimen Selesai! Hasil disimpan di hasil_variasi_model.csv")


Memulai Training: SimpleRNN_Shallow_Small_L1_H128

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Image_Input (InputLayer)    [(None, 2048)]               0         []                            
                                                                                                  
 dense (Dense)               (None, 256)                  524544    ['Image_Input[0][0]']         
                                                                                                  
 reshape (Reshape)           (None, 1, 256)               0         ['dense[0][0]']               
                                                                                                  
 Caption_Input (InputLayer)  [(None, 35)]                 0         []                            
                                        

KeyboardInterrupt: 

### Bagian 4

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import kelas dari file teman Anda
from ImageCaptioningScratch import ImageCaptioningModel

eval_model = build_model(
    lstm=True, 
    layers=1, 
    hidden_state_count=128, 
    vocab_size=VOCAB_SIZE, 
    sequence_length=SEQ_LEN
)

eval_model.load_weights("weights/LSTM_Shallow_Small_L1_H128.h5")

test_image_path = "../../dataset/Images/1000268201_693b08cb0e.jpg"

model_manual = ImageCaptioningModel(
    keras_model=eval_model,
    text_util=text_util,
    is_lstm=True
)

Model: "model_16"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Image_Input (InputLayer)    [(None, 2048)]               0         []                            
                                                                                                  
 dense_14 (Dense)            (None, 256)                  524544    ['Image_Input[0][0]']         
                                                                                                  
 reshape_14 (Reshape)        (None, 1, 256)               0         ['dense_14[0][0]']            
                                                                                                  
 Caption_Input (InputLayer)  [(None, 35)]                 0         []                            
                                                                                           

In [ ]:
import os

nama_file = os.path.basename(test_image_path)

fitur_gambar = image_features_map[nama_file]
print(list(text_util.word_to_idx.keys())[:20])

hasil_kalimat = model_manual.generate_caption(image_feature=fitur_gambar, max_len=SEQ_LEN)

tokens_to_remove = ["<unk>", "<start>", "<end>"]
hasil_kalimat_bersih = ' '.join([word for word in hasil_kalimat.split() if word not in tokens_to_remove])

print("Hasil prediksi kalimat:", hasil_kalimat_bersih)

['<unk>', 'a', 'end', 'start', 'in', 'the', 'on', 'is', 'and', 'dog', 'with', 'man', 'of', 'two', 'white', 'black', 'boy', 'are', 'woman', 'girl']
Hasil prediksi kalimat: a man in a dog


In [ ]:
from nltk.translate.bleu_score import sentence_bleu


hipotesis = hasil_kalimat_bersih.split()


nama_file = os.path.basename(test_image_path)  
referensi = map_image_cap.get(nama_file, []) 

referensi = [ref.split() for ref in referensi]

skor_bleu1 = sentence_bleu(referensi, hipotesis, weights=(1.0, 0, 0, 0))
print("Skor BLEU-1:", skor_bleu1)

skor_bleu4 = sentence_bleu(referensi, hipotesis, weights=(0.25, 0.25, 0.25, 0.25))
print("Skor BLEU-4:", skor_bleu4)

Skor BLEU-1: 0.26959737847033294
Skor BLEU-4: 4.1711849316776384e-155


d:\tubes2_ml\venv\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
d:\tubes2_ml\venv\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [ ]:
from nltk.translate.bleu_score import sentence_bleu

def evaluasi_model_massal(model_manual, daftar_gambar_uji, image_features_map, map_image_cap, text_util, max_len):
    total_bleu1 = 0
    total_bleu4 = 0
    jumlah_data = len(daftar_gambar_uji)
    
    tokens_to_remove = ["<unk>", "start", "end"]
    
    for nama_file in daftar_gambar_uji:
        fitur_gambar = image_features_map.get(nama_file)
        if fitur_gambar is None:
            print(f"Fitur gambar untuk {nama_file} tidak ditemukan, melewati...")
            jumlah_data -= 1
            continue
        
        hasil_kalimat = model_manual.generate_caption(image_feature=fitur_gambar, max_len=max_len)
        
        hasil_kalimat_bersih = ' '.join([word for word in hasil_kalimat.split() if word not in tokens_to_remove])
        
        referensi = map_image_cap.get(nama_file, [])
        referensi = [ref.split() for ref in referensi]
        
        if not referensi:
            print(f"Tidak ada referensi untuk {nama_file}, melewati...")
            jumlah_data -= 1
            continue
        
        skor_bleu1_gambar_ini = sentence_bleu(referensi, hasil_kalimat_bersih.split(), weights=(1.0, 0, 0, 0))
        skor_bleu4_gambar_ini = sentence_bleu(referensi, hasil_kalimat_bersih.split(), weights=(0.25, 0.25, 0.25, 0.25))
        
        total_bleu1 += skor_bleu1_gambar_ini
        total_bleu4 += skor_bleu4_gambar_ini
    
    if jumlah_data > 0:
        rata_rata_bleu1 = total_bleu1 / jumlah_data
        rata_rata_bleu4 = total_bleu4 / jumlah_data
    else:
        rata_rata_bleu1 = 0
        rata_rata_bleu4 = 0
    
    return rata_rata_bleu1, rata_rata_bleu4

In [ ]:
rata_rata_bleu1, rata_rata_bleu4 = evaluasi_model_massal(
    model_manual=model_manual,
    daftar_gambar_uji=test_keys,
    image_features_map=image_features_map,
    map_image_cap=map_image_cap,
    text_util=text_util,
    max_len=SEQ_LEN
)

print(f"Rata-rata Skor BLEU-1: {rata_rata_bleu1:.4f}")
print(f"Rata-rata Skor BLEU-4: {rata_rata_bleu4:.4f}")

Rata-rata Skor BLEU-1: 0.3346
Rata-rata Skor BLEU-4: 0.0230
